# Qwen2.5-7B FRAML Fine-Tune → Ollama (vast.ai)

**Pipeline:**
1. Install Unsloth + deps
2. QLoRA fine-tune on `framl_train.jsonl` (tool-calling)
3. Export merged model to GGUF (q4_k_m) via Unsloth
4. Register with Ollama + start serving on `0.0.0.0:11434`
5. Point Dash app at `http://<vast-ip>:<port>/v1`

**Recommended vast.ai instance:** RTX 3090 24GB or RTX 4090 24GB (~\$0.30-0.50/hr)  
**Expose port 11434** when creating the instance so the Dash app can connect.

## Cell 1 — Install Dependencies
Unsloth must be installed before transformers/trl.

In [ ]:
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q torch>=2.1.0 transformers>=4.46.0 trl>=0.12.0 peft>=0.13.0 \
              datasets>=3.0.0 bitsandbytes>=0.44.0 accelerate>=0.34.0 \
              sentencepiece huggingface_hub

## Cell 2 — Environment & Imports
`HF_HOME` must be set **before** any HuggingFace import — libraries cache the path at import time.

In [ ]:
import os

# vast.ai: root disk is small (~16 GB). Redirect HF cache to RAM-backed /dev/shm
os.environ["HF_HOME"] = "/dev/shm/hf_cache"

import json
import torch
from pathlib import Path
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 3 — Configuration
Adjust `EPOCHS` and `BATCH_SIZE` for your GPU. RTX 3090/4090 (24 GB): batch=2, accum=4 is safe.

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────
BASE_MODEL   = "unsloth/Qwen2.5-7B-Instruct"   # Unsloth 4-bit quantised hub mirror
MAX_SEQ_LEN  = 4096

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

# ── Training ───────────────────────────────────────────────────────────────
EPOCHS      = 5       # 60 examples x 5 epochs (~5-10 min on RTX 3090)
BATCH_SIZE  = 2
GRAD_ACCUM  = 4       # effective batch = 8
LR          = 2e-4

# ── Paths ──────────────────────────────────────────────────────────────────
# Works whether you run from finetune/ or repo root
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "finetune":
    REPO_ROOT = REPO_ROOT.parent
DATA_FILE = REPO_ROOT / "finetune" / "data" / "framl_train.jsonl"
OUT_DIR   = Path("/dev/shm/qwen-framl")         # save adapter to RAM disk
GGUF_DIR  = Path("/dev/shm/qwen-framl-gguf")    # merged HF safetensors (temp)
F16_GGUF  = Path("/dev/shm/qwen-framl-f16.gguf")
Q4KM_GGUF = Path("/dev/shm/qwen-framl-q4km.gguf")

OUT_DIR.mkdir(parents=True, exist_ok=True)
GGUF_DIR.mkdir(parents=True, exist_ok=True)

# ── HuggingFace Hub (for saving GGUF mid-session) ──────────────────────────
# Set HF_TOKEN before running: export HF_TOKEN=hf_xxxxxxxxxxxx
HF_REPO  = "speri420/qwen-framl"
HF_TOKEN = os.environ.get("HF_TOKEN", "")

# ── Ollama ─────────────────────────────────────────────────────────────────
OLLAMA_MODEL_NAME = "qwen-framl"

print(f"Data:    {DATA_FILE}  (exists={DATA_FILE.exists()})")
print(f"Adapter: {OUT_DIR}")
print(f"GGUF:    {Q4KM_GGUF}")
print(f"HF repo: {HF_REPO}  (token set={bool(HF_TOKEN)})")

## Cell 4 — Load Base Model

In [ ]:
print(f"Loading {BASE_MODEL} in 4-bit QLoRA ...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,          # auto (bfloat16 on Ampere+)
)

# Apply Qwen2.5 chat template (includes tool-call special tokens)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

print("Model loaded.")

In [ ]:
import shutil as _shutil

hf_cache = Path("/dev/shm/hf_cache")
if hf_cache.exists():
    _shutil.rmtree(hf_cache)
    print("HF cache cleared — /dev/shm freed for GGUF export.")
else:
    print("HF cache already gone.")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,}  ({100*trainable/total:.2f}% of {total:,})")

## Cell 6 — Load & Format Dataset

In [ ]:
def load_jsonl(path):
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return Dataset.from_list(records)


def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


raw_ds       = load_jsonl(DATA_FILE)
formatted_ds = raw_ds.map(format_example, remove_columns=raw_ds.column_names)

print(f"Training examples: {len(formatted_ds)}")
print("\n--- Sample (first 500 chars) ---")
print(formatted_ds[0]["text"][:500])

## Cell 7 — Train

In [ ]:
sft_config = SFTConfig(
    output_dir=str(OUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    save_strategy="no",   # skip mid-run checkpoints to save disk
    report_to="none",
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_ds,
    args=sft_config,
)

print("Starting training ...")
stats = trainer.train()
print(f"\nTraining complete. Final loss: {stats.training_loss:.4f}")

## Cell 8 — Export to GGUF (q4_k_m)

Manual step-by-step conversion — more reliable than Unsloth's built-in wrapper (which hides errors via `capture_output=True`).

**Pipeline:** merge LoRA → save HF safetensors → convert to f16 GGUF → quantize to q4_k_m → clean up  
**Space needed:** ~15 GB free in `/dev/shm` at start (HF cache cleared in Cell 4b)  
**Time:** ~10–15 min on RTX 3090

In [ ]:
import subprocess, glob as _glob

LLAMA_CPP = Path("/root/.unsloth/llama.cpp")
CONVERTER = LLAMA_CPP / "unsloth_convert_hf_to_gguf.py"
QUANTIZER = LLAMA_CPP / "llama-quantize"

# Step 1 — Merge LoRA into base weights, save as HF safetensors
print("Step 1/4 — Merging LoRA and saving safetensors to", GGUF_DIR, "...")
model.save_pretrained_merged(str(GGUF_DIR), tokenizer, save_method="merged_16bit")
print("Safetensors saved.")

# Step 2 — Convert HF safetensors → f16 GGUF
print(f"\nStep 2/4 — Converting to f16 GGUF: {F16_GGUF} ...")
result = subprocess.run(
    ["python", str(CONVERTER), "--outfile", str(F16_GGUF), "--outtype", "f16", str(GGUF_DIR)],
    check=True
)
print(f"f16 GGUF written: {F16_GGUF.stat().st_size / 1e9:.1f} GB")

# Step 3 — Delete safetensors to reclaim /dev/shm space
print("\nStep 3/4 — Removing safetensors ...")
for sf in _glob.glob(str(GGUF_DIR / "model-*.safetensors")):
    Path(sf).unlink()
    print(f"  deleted {sf}")

# Step 4 — Quantize f16 → q4_k_m  (cannot skip straight from safetensors)
print(f"\nStep 4/4 — Quantizing to q4_k_m: {Q4KM_GGUF} ...")
subprocess.run(
    [str(QUANTIZER), str(F16_GGUF), str(Q4KM_GGUF), "Q4_K_M"],
    check=True
)
print(f"q4_k_m GGUF written: {Q4KM_GGUF.stat().st_size / 1e9:.1f} GB")

# Clean up f16 (no longer needed)
F16_GGUF.unlink()
print(f"\nDeleted f16 GGUF. Final: {Q4KM_GGUF}  ({Q4KM_GGUF.stat().st_size / 1e9:.1f} GB)")

import subprocess, glob as _glob, shutil as _shutil, sys

LLAMA_CPP = Path("/root/.unsloth/llama.cpp")
CONVERTER = LLAMA_CPP / "unsloth_convert_hf_to_gguf.py"
QUANTIZER = LLAMA_CPP / "llama-quantize"

# Step 1 — Merge LoRA into base weights, save as HF safetensors
print("Step 1/4 — Merging LoRA and saving safetensors to", GGUF_DIR, "...")
model.save_pretrained_merged(str(GGUF_DIR), tokenizer, save_method="merged_16bit")
print("Safetensors saved.")

# Step 2 — Convert HF safetensors → f16 GGUF
print(f"\nStep 2/4 — Converting to f16 GGUF: {F16_GGUF} ...")
subprocess.run(
    [sys.executable, str(CONVERTER), "--outfile", str(F16_GGUF), "--outtype", "f16", str(GGUF_DIR)],
    check=True
)
print(f"f16 GGUF written: {F16_GGUF.stat().st_size / 1e9:.1f} GB")

# Step 3 — Delete safetensors to reclaim /dev/shm space
print("\nStep 3/4 — Removing safetensors ...")
for sf in _glob.glob(str(GGUF_DIR / "model-*.safetensors")):
    Path(sf).unlink()
    print(f"  deleted {sf}")

# Step 4 — Quantize f16 → q4_k_m
print(f"\nStep 4/4 — Quantizing to q4_k_m: {Q4KM_GGUF} ...")
subprocess.run(
    [str(QUANTIZER), str(F16_GGUF), str(Q4KM_GGUF), "Q4_K_M"],
    check=True
)
print(f"q4_k_m GGUF written: {Q4KM_GGUF.stat().st_size / 1e9:.1f} GB")

# Clean up f16 and HF cache (no longer needed)
F16_GGUF.unlink()
hf_cache = Path("/dev/shm/hf_cache")
if hf_cache.exists():
    _shutil.rmtree(hf_cache)
    print("HF cache cleared.")

print(f"\nFinal GGUF: {Q4KM_GGUF}  ({Q4KM_GGUF.stat().st_size / 1e9:.1f} GB)")

In [ ]:
from huggingface_hub import HfApi

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN not set. In the vast.ai terminal run:\n"
        "  export HF_TOKEN=hf_xxxxxxxxxxxx\n"
        "then re-run Cell 3 to pick it up."
    )
if HF_REPO == "your-hf-username/qwen-framl":
    raise ValueError("Set HF_REPO to your actual HuggingFace repo name in Cell 3.")
if not Q4KM_GGUF.exists():
    raise FileNotFoundError(f"{Q4KM_GGUF} not found — run Cell 8 first.")

api = HfApi()

# Create repo if it doesn't exist yet
api.create_repo(repo_id=HF_REPO, token=HF_TOKEN, repo_type="model", exist_ok=True)
print(f"Repo: https://huggingface.co/{HF_REPO}")

# Upload the GGUF (resumes automatically if interrupted)
print(f"Uploading {Q4KM_GGUF.name}  ({Q4KM_GGUF.stat().st_size / 1e9:.1f} GB) ...")
api.upload_file(
    path_or_fileobj=str(Q4KM_GGUF),
    path_in_repo="qwen-framl-q4km.gguf",
    repo_id=HF_REPO,
    token=HF_TOKEN,
    repo_type="model",
)
print(f"\nUpload complete. Pull it later with:")
print(f"  huggingface-cli download {HF_REPO} qwen-framl-q4km.gguf")

## Cell 9 — Install Ollama

In [ ]:
import subprocess, shutil

if shutil.which("ollama") is None:
    print("Installing Ollama ...")
    result = subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True, capture_output=True, text=True
    )
    print(result.stdout[-2000:] if result.stdout else "")
    if result.returncode != 0:
        print("STDERR:", result.stderr[-1000:])
    else:
        print("Ollama installed.")
else:
    v = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
    print(f"Ollama already installed: {v.stdout.strip()}")

## Cell 10 — Write Modelfile & Register with Ollama
The Modelfile embeds the FRAML system prompt so every chat session starts in the right context.

In [ ]:
import subprocess

if not Q4KM_GGUF.exists():
    raise FileNotFoundError(f"{Q4KM_GGUF} not found — run Cell 8 first.")

print(f"Using GGUF: {Q4KM_GGUF}  ({Q4KM_GGUF.stat().st_size / 1e9:.1f} GB)")

SYSTEM_PROMPT = (
    "You are a FRAML (Fraud + AML) analytics AI assistant. "
    "You analyze false positive/false negative trade-offs in AML alert thresholds, "
    "perform customer behavioral segmentation, and interpret clustering results. "
    "Use the available tools to retrieve data, then provide clear, analytical insights. "
    "Be concise and reference specific numbers when interpreting results."
)

modelfile_content = f"""FROM {Q4KM_GGUF}

PARAMETER num_ctx 4096
PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER stop "<|im_end|>"

SYSTEM "{SYSTEM_PROMPT}"
"""

modelfile_path = Path("/tmp/Modelfile.framl")
modelfile_path.write_text(modelfile_content)
print("--- Modelfile ---")
print(modelfile_content)

In [ ]:
print(f"Registering '{OLLAMA_MODEL_NAME}' with Ollama ...")
result = subprocess.run(
    ["ollama", "create", OLLAMA_MODEL_NAME, "-f", str(modelfile_path)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)
else:
    print(f"Model '{OLLAMA_MODEL_NAME}' registered.")

## Cell 11 — Start Ollama Server
Binds to `0.0.0.0:11434` so your local Dash app can reach it via the vast.ai public IP + exposed port.

In [ ]:
import time

# Kill any existing instance
subprocess.run("pkill -f 'ollama serve' || true", shell=True)
time.sleep(2)

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"

server_proc = subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

print("Waiting for Ollama to start ...")
time.sleep(5)

check = subprocess.run(["ollama", "list"], capture_output=True, text=True)
if check.returncode == 0:
    print("Ollama is running. Available models:")
    print(check.stdout)
else:
    print("Ollama may still be starting — wait a few seconds and re-run this cell.")

## Cell 12 — Smoke Test

In [ ]:
import urllib.request, json as _json

payload = _json.dumps({
    "model": OLLAMA_MODEL_NAME,
    "messages": [
        {"role": "user", "content": "Show me threshold tuning for individual customers"}
    ],
    "stream": False,
    "tools": [
        {
            "type": "function",
            "function": {
                "name": "threshold_tuning",
                "description": "Compute FP/FN trade-off across threshold range",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "customer_type": {"type": "string", "enum": ["Business", "Individual"]},
                        "threshold_min": {"type": "number"},
                        "threshold_max": {"type": "number"},
                        "steps": {"type": "integer"}
                    },
                    "required": ["customer_type"]
                }
            }
        }
    ]
}).encode()

req = urllib.request.Request(
    "http://localhost:11434/v1/chat/completions",
    data=payload,
    headers={"Content-Type": "application/json"}
)

with urllib.request.urlopen(req, timeout=60) as resp:
    result = _json.loads(resp.read())

msg = result["choices"][0]["message"]
if msg.get("tool_calls"):
    tc = msg["tool_calls"][0]
    print(f"Tool called:  {tc['function']['name']}")
    print(f"Arguments:    {tc['function']['arguments']}")
    print("\nSMOKE TEST PASSED")
else:
    print("WARNING: no tool call. Model replied with text:")
    print(msg.get("content", ""))

## Cell 13 — Connect Your Dash App

On your **local Windows machine**, set env vars before starting `application.py`:

**CMD:**
```bat
set OLLAMA_BASE_URL=http://<vast-ip>:<mapped-port>/v1
set OLLAMA_MODEL=qwen-framl
.venv\Scripts\python.exe application.py
```

**PowerShell:**
```powershell
$env:OLLAMA_BASE_URL = "http://<vast-ip>:<mapped-port>/v1"
$env:OLLAMA_MODEL    = "qwen-framl"
.venv\Scripts\python.exe application.py
```

Find `<vast-ip>` and `<mapped-port>` in your vast.ai instance dashboard under **"Open Ports"**.

---

### Keep Ollama alive after closing the notebook

Run this in a vast.ai terminal so it survives kernel restarts:

```bash
OLLAMA_HOST=0.0.0.0:11434 nohup ollama serve > /tmp/ollama.log 2>&1 &
ollama list   # confirm qwen-framl is listed
```